# IVG v2.0.0 — Biomedical Knowledge Graph at Scale (DRKG)

This notebook runs InterSystems IRIS Vector Graph against the **Drug Repurposing Knowledge Graph** ([DRKG](https://github.com/gnn4dr/DRKG), Apache-2.0):

- **97,238 nodes** across 13 biomedical types (Gene, Compound, Disease, Anatomy, Pathway, ...)
- **5,874,261 edges** (107 relation types from 6 source databases)
- **400-dim TransE knowledge-graph embeddings** (optional, for hybrid search)

It demonstrates the v2.0.0 graph-analytics suite end-to-end on a real biomedical graph: **centrality** (which genes are most influential?), **community detection** (which drugs/genes/diseases cluster together?), and **hybrid vector+graph** retrieval.

> Prereq: load DRKG into the `ivg-iris` container first:
> ```bash
> python scripts/load_drkg.py --embeddings
> ```

## 1. Connect and inspect scale

In [ ]:
import iris
from iris_vector_graph.engine import IRISGraphEngine

# Connect to ivg-iris (use the container IP or localhost as appropriate)
conn = iris.connect('localhost', 1972, 'USER', '_SYSTEM', 'SYS')
engine = IRISGraphEngine(conn, embedding_dimension=400)

status = engine.status()
print(status)

The concept-first status (spec 180) shows graph size, vector/full-text index state, and sync state — without exposing internal `^`-globals. For IRIS-developer internals, use `engine.status(internals=True)`.

## 2. Centrality — which genes are most influential?

Degree centrality ranks nodes by connection count. On DRKG, the top genes by out-degree are the most-studied / most-connected in the literature graph.

In [ ]:
deg = engine.degree_centrality(direction='both', top_k=15)
print('Top 15 nodes by degree centrality:')
for r in deg:
    print(f"  {r['id']:<28}  degree={r['score']:.0f}")

**Betweenness** finds bridge nodes — entities that connect otherwise-separate regions of the graph (often multi-pathway genes or broad-spectrum compounds). Uses sampled Brandes for tractability at this scale.

In [ ]:
btw = engine.betweenness_centrality(sample_size=500, top_k=15)
print('Top 15 bridge nodes (sampled betweenness):')
for r in btw:
    print(f"  {r['id']:<28}  betweenness={r['score']:.4f}")

## 3. Community detection — drug/gene/disease clusters

Leiden finds densely-connected communities. On a container with `igraph`+`leidenalg` in embedded Python, this runs the **canonical** leidenalg algorithm server-side (spec 185); otherwise it falls back to a pure-ObjectScript greedy partition. Both return the same shape.

In [ ]:
comms = engine.leiden_communities(random_seed=42, top_k=0)
from collections import Counter
sizes = Counter(c['community'] for c in comms)
print(f'Found {len(sizes)} communities over {len(comms):,} nodes')
print('Largest 10 communities (id: size):')
for cid, sz in sizes.most_common(10):
    print(f'  community {cid}: {sz:,} nodes')

In [ ]:
# Inspect the node-type composition of the largest community
biggest = sizes.most_common(1)[0][0]
members = [c['id'] for c in comms if c['community'] == biggest]
type_mix = Counter(m.split('::', 1)[0] for m in members)
print(f'Largest community ({len(members):,} nodes) type mix:')
for t, n in type_mix.most_common():
    print(f'  {t:<22} {n:,}')

## 4. Hybrid vector + graph — find genes like TP53 near a disease

The canonical drug-repurposing query: *given a disease, find genes similar to a known target within N hops.* This fuses TransE vector similarity with graph proximity. (Requires `--embeddings` at load time.)

In [ ]:
# Requires embeddings loaded. TP53 = Gene::7157 in DRKG.
if engine.status().ready_for_vector_search:
    tp53 = engine.get_embedding('Gene::7157')
    if tp53:
        similar = engine.search_nodes_by_vector(tp53['embedding'], top_k=10)
        print('Genes/entities most similar to TP53 (TransE cosine):')
        for r in similar:
            print(f"  {r.get('id','?'):<28} score={r.get('score',0):.3f}")
    else:
        print('TP53 embedding not found — check entity id / load --embeddings')
else:
    print('No vector index — run scripts/load_drkg.py --embeddings to enable this section')

## 5. Cypher — the same analytics from a query

Every algorithm is also a Cypher procedure (`CALL ivg.*`), so the same analytics run from the query surface (and server-side via the SQL function path, no Python client required).

In [ ]:
result = engine.execute_cypher(
    "CALL ivg.degreeCentrality({topK: 10}) YIELD node, score, degree "
    "RETURN node, score, degree ORDER BY score DESC LIMIT 10"
)
print('columns:', result.columns)
for row in result.rows:
    print(' ', row)

---

**What this demonstrates for v2.0.0:** the full graph-analytics suite (degree/betweenness/closeness/eigenvector centrality + leiden/triangle/scc/kcore community detection + hybrid vector search) running on a **97K-node / 5.87M-edge real biomedical KG** — via both the Python API and the Cypher `CALL` surface. See `docs/performance/DRKG_SCALE.md` for recorded timings.